### Importing required libraries

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from datasets import load_dataset

### Loading dataset

In [ ]:
raw_data = load_dataset("wikitext", "wikitext-103-raw-v1", split="train", streaming=True)

text_list = []
line_count = 0

for entry in raw_data:
    content = entry['text'].strip().lower()

    if len(content) > 30 and not content.startswith('='):
        text_list.append(content)
        line_count += 1
    if line_count > 8000:
        break

print(f"Collected {len(text_list)} clean English sentences.")

Collected 8001 clean English sentences.


### Tokenization

In [ ]:
vocab_size = 8000
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(text_list)
total_words = len(tokenizer.word_index) + 1

### Convert Text to Sequence of integers

In [ ]:
input_sequences = []
for line in text_list:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram = token_list[max(0, i-5):i+1]
        input_sequences.append(n_gram)

### Zero padding

In [ ]:
X_Y = np.array(pad_sequences(input_sequences, maxlen=6, padding='pre'), dtype='int32')

### Creating X and y

In [ ]:
X = X_Y[:, :-1]
y = X_Y[:, -1]

print(f"Total sequences created: {len(X)}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

Total sequences created: 835217
X shape: (835217, 5), y shape: (835217,)


### Building a LSTM Model

In [ ]:
model = Sequential([
    Embedding(total_words, 64, input_shape=(5,)),
    LSTM(128, return_sequences=False),
    Dropout(0.3),
    Dense(total_words, activation='softmax')
])

In [ ]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 5, 64)          │     2,786,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 43537)          │     5,616,273 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,501,457 (32.43 MB)

 Trainable params: 8,501,457 (32.43 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

In [ ]:
history = model.fit(X, y, epochs=30, batch_size=512, validation_split=0.2, callbacks=callbacks)

Epoch 1/30
1306/1306 ━━━━━━━━━━━━━━━━━━━━ 78s 58ms/step - accuracy: 0.1171 - loss: 6.5365 - val_accuracy: 0.1439 - val_loss: 6.4173 - learning_rate: 0.0010
Epoch 2/30
1306/1306 ━━━━━━━━━━━━━━━━━━━━ 75s 57ms/step - accuracy: 0.1524 - loss: 6.0512 - val_accuracy: 0.1551 - val_loss: 6.2624 - learning_rate: 0.0010
Epoch 3/30
1306/1306 ━━━━━━━━━━━━━━━━━━━━ 75s 57ms/step - accuracy: 0.1645 - loss: 5.8767 - val_accuracy: 0.1633 - val_loss: 6.1567 - learning_rate: 0.0010
Epoch 4/30
1306/1306 ━━━━━━━━━━━━━━━━━━━━ 82s 63ms/step - accuracy: 0.1719 - loss: 5.7447 - val_accuracy: 0.1683 - val_loss: 6.0767 - learning_rate: 0.0010
Epoch 5/30
1306/1306 ━━━━━━━━━━━━━━━━━━━━ 75s 58ms/step - accuracy: 0.1783 - loss: 5.6374 - val_accuracy: 0.1725 - val_loss: 6.0294 - learning_rate: 0.0010
Epoch 6/30
1306/1306 ━━━━━━━━━━━━━━━━━━━━ 75s 58ms/step - accuracy: 0.1823 - loss: 5.5484 - val_accuracy: 0.1756 - val_loss: 5.9930 - learning_rate: 0.0010
Epoch 7/30
1306/1306 ━━━━━━━━━━━━━━━━━━━━ 75s 58ms/step - accura

### Prediction

In [ ]:
def predict_next(seed_text, next_words=1):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=10, padding='pre')

        probs = model.predict(token_list, verbose=0)

        predicted_idx = np.argmax(probs, axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_idx:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

print(predict_next("the most important part of the", next_words=5))

the most important part of the <OOV> <OOV> <OOV> <OOV> <OOV>


In [ ]:
seed = "the research showed that"
next_words = 10

for _ in range(next_words):
    next_w = predict_next(seed)
    seed += " " + next_w

print(f"Generated Sentence: {seed}")

Generated Sentence: the research showed that the research showed that the the research showed that the research showed that the <OOV> the research showed that the research showed that the the research showed that the research showed that the <OOV> of the research showed that the research showed that the the research showed that the research showed that the <OOV> the research showed that the research showed that the the research showed that the research showed that the <OOV> of the the research showed that the research showed that the the research showed that the research showed that the <OOV> the research showed that the research showed that the the research showed that the research showed that the <OOV> of the research showed that the research showed that the the research showed that the research showed that the <OOV> the research showed that the research showed that the the research showed that the research showed that the <OOV> of the <OOV> the research showed that the research show